# 05 — Land Cover & Topographic DEM Analysis
## RoadSafe Delhi: Real Physical Environmental Feature Extraction
### Resolves Issue 1: Replaces synthetic formulas with real raster extraction from NASA SRTM DEM and ESA WorldCover 10m.


In [ ]:
import os
import sys

# Robust project-relative path configuration
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
DATASETS_DIR = os.path.join(PROJECT_ROOT, "Datasets")
DATA_PROC_DIR = os.path.join(PROJECT_ROOT, "data_processed")
DATA_ENG_DIR = os.path.join(PROJECT_ROOT, "data_engineered")
DATA_FINAL_DIR = os.path.join(PROJECT_ROOT, "data_final")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
EDA_DIR = os.path.join(OUTPUTS_DIR, "eda")
MAPS_DIR = os.path.join(OUTPUTS_DIR, "maps")
TABLES_DIR = os.path.join(OUTPUTS_DIR, "tables")
REPORTS_DIR = os.path.join(OUTPUTS_DIR, "reports")

print("Project Root:", PROJECT_ROOT)


### 1. NASA SRTM 30m DEM Elevation & Slope


In [ ]:
import rasterio
from rasterio.merge import merge
import numpy as np

dem_cache = os.path.join(PROJECT_ROOT, ".cache", "dem")
src1 = rasterio.open(os.path.join(dem_cache, "N28E076.hgt"))
src2 = rasterio.open(os.path.join(dem_cache, "N28E077.hgt"))
mosaic, transform = merge([src1, src2])
dem_arr = mosaic[0]

res_y = abs(transform[4]) * 111320.0
res_x = abs(transform[0]) * 111320.0 * np.cos(np.radians(28.6))
gy, gx = np.gradient(dem_arr.astype(float), res_y, res_x)
slope_arr = np.degrees(np.arctan(np.sqrt(gx**2 + gy**2)))

print(f"DEM Grid Shape: {dem_arr.shape}")
print(f"Elevation Range: {dem_arr[dem_arr > -1000].min()}m to {dem_arr[dem_arr > -1000].max()}m")
print(f"Slope Range:     {slope_arr.min():.2f}° to {slope_arr.max():.2f}° (Median: {np.median(slope_arr):.2f}°)")


### 2. ESA WorldCover 10m Built-Up Density


In [ ]:
from rasterio.windows import from_bounds
tif_path = os.path.join(DATASETS_DIR, "ESA_WorldCover_10m_2021_v200_N27E075_Map.tif")
with rasterio.open(tif_path) as src:
    w = from_bounds(76.80, 28.35, 77.45, 28.95, transform=src.transform)
    delhi_lc = src.read(1, window=w)

builtup_pct = (delhi_lc == 50).mean() * 100.0
print(f"Delhi Regional Built-Up (Urban) Land Cover: {builtup_pct:.2f}%")


### 3. Land Cover & Elevation Profiles


In [ ]:
from IPython.display import Image, display
display(Image(filename=os.path.join(EDA_DIR, "delhi_landcover_distribution.png")))
display(Image(filename=os.path.join(EDA_DIR, "delhi_elevation_slope_profile.png")))
